In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gdown
import numpy as np

In [3]:
!gdown --id 1kyCMfQxflrIwYeJnc0MprYEdre7Qu83k -O X_train.npy

!gdown --id 1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4 -O X_test.npy

!gdown --id 1mF6szu7juZ5n75xOix5CfwwpbpklJIDj -O X_val.npy

!gdown --id 1WwnlFTAZ36W-A-rrvOnq0KzFusdFbYd3 -O y_train.npy

!gdown --id 1qL6KE1XlJmF6epaKzoETb-nOiesZsr39 -O y_test.npy

!gdown --id 1sU9NmM_-qLERVlf02_03Q0yXi3QVg6AP -O y_val.npy

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1kyCMfQxflrIwYeJnc0MprYEdre7Qu83k
From (redirected): https://drive.google.com/uc?id=1kyCMfQxflrIwYeJnc0MprYEdre7Qu83k&confirm=t&uuid=7994020f-113a-45b4-a48e-f2a282d4c81f
To: /content/X_train.npy
100% 2.19G/2.19G [00:34<00:00, 63.2MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4
From (redirected): https://drive.google.com/uc?id=1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4&confirm=t&uuid=e3f0a599-24b6-4332-8bdc-7ae0aecb916d
To: /content/X_test.npy
10

In [4]:
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

X_train shape: (57788, 256, 37)
y_train shape: (57788,)


In [5]:
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

X_test shape: (5000, 256, 37)
y_test shape: (5000,)


In [6]:
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')

print('X_val shape:', X_val.shape)
print('y_val shape:', y_val.shape)

X_val shape: (5000, 256, 37)
y_val shape: (5000,)


In [7]:
print("Train OCD:", sum(y_train == 1))
print("Train Non-OCD:", sum(y_train == 0))

Train OCD: 20000
Train Non-OCD: 37788


In [8]:
X_ocd = X_train[y_train == 1]
y_ocd = y_train[y_train == 1]

print("OCD samples:", X_ocd.shape)

OCD samples: (20000, 256, 37)


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(256, 37)))
model.add(Dropout(0.3))

model.add(LSTM(32))
model.add(Dropout(0.3))

model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
class_weight = {0: 1, 1: 3}

In [11]:
X_train = X_train.astype('float32')
X_val = X_val.astype('float32')
X_test = X_test.astype('float32')

y_train = y_train.astype('int32')
y_val = y_val.astype('int32')
y_test = y_test.astype('int32')

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 256, 64)        │        26,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,073 (152.63 KB)

 Trainable params: 39,073 (152.63 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/25
1806/1806 ━━━━━━━━━━━━━━━━━━━━ 47s 23ms/step - accuracy: 0.3612 - loss: 1.1176 - val_accuracy: 0.2354 - val_loss: 0.8811
Epoch 2/25
1806/1806 ━━━━━━━━━━━━━━━━━━━━ 42s 23ms/step - accuracy: 0.6119 - loss: 0.9638 - val_accuracy: 0.6312 - val_loss: 0.7070
Epoch 3/25
1806/1806 ━━━━━━━━━━━━━━━━━━━━ 41s 23ms/step - accuracy: 0.7987 - loss: 0.6720 - val_accuracy: 0.8066 - val_loss: 0.4724
Epoch 4/25
1806/1806 ━━━━━━━━━━━━━━━━━━━━ 41s 23ms/step - accuracy: 0.8673 - loss: 0.4826 - val_accuracy: 0.8070 - val_loss: 0.4919
Epoch 5/25
1806/1806 ━━━━━━━━━━━━━━━━━━━━ 81s 22ms/step - accuracy: 0.9029 - loss: 0.3750 - val_accuracy: 0.7878 - val_loss: 0.6062
Epoch 6/25
1806/1806 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - accuracy: 0.9243 - loss: 0.3007 - val_accuracy: 0.8606 - val_loss: 0.4303
Epoch 7/25
1806/1806 ━━━━━━━━━━━━━━━━━━━━ 40s 22ms/step - accuracy: 0.9373 - loss: 0.2539 - val_accuracy: 0.8344 - val_loss: 0.5528
Epoch 8/25
1806/1806 ━━━━━━━━━━━━━━━━━━━━ 40s 22ms/step - accuracy: 0.9477 -

In [18]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_test) > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
[[4264  460]
 [ 248   28]]
              precision    recall  f1-score   support

           0       0.95      0.90      0.92      4724
           1       0.06      0.10      0.07       276

    accuracy                           0.86      5000
   macro avg       0.50      0.50      0.50      5000
weighted avg       0.90      0.86      0.88      5000



In [19]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8584 - loss: 0.4435
Test Accuracy: 0.8583999872207642
Test Loss: 0.44353780150413513


In [20]:
print("Training Accuracy:", history.history['accuracy'][-1])

Training Accuracy: 0.9601820707321167
